In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
!pip install kagglehub
import kagglehub
swaptr_fifa_wc_2026_players_path = kagglehub.dataset_download('swaptr/fifa-wc-2026-players')
swaptr_fifa_wc_2026_teams_path = kagglehub.dataset_download('swaptr/fifa-wc-2026-teams')
swaptr_fifa_wc_2026_matches_path = kagglehub.dataset_download('swaptr/fifa-wc-2026-matches')

print('Data source import complete.')


### Copa Mundial de la FIFA 2026: Análisis Exploratorio de Datos

La primera Copa Mundial con **48 selecciones** y la primera organizada por **tres países** (Estados Unidos, Canadá y México) comenzó el **11 de junio de 2026** en el renovado **Estadio Azteca** (actualmente con el nombre comercial **Estadio Banorte**). Este cuaderno realiza un seguimiento del torneo a medida que avanza: en cada ejecución vuelve a cargar las tablas de partidos, equipos y jugadores, y recalcula todas las métricas y visualizaciones a partir de los datos, garantizando que el análisis se mantenga actualizado desde la fase inicial hasta la final.

El análisis se basa en tres tablas principales:

* **`matches`**: contiene un registro por cada partido disputado, incluyendo marcador, posesión del balón, tiros, tarjetas, faltas, jugadas a balón parado, formaciones, entrenadores, capitanes, estadio, árbitro y asistencia. Los encuentros programados que aún no se han jugado aparecen con estadísticas vacías y se excluyen automáticamente de las gráficas basadas en resultados.
* **`teams`**: contiene un registro por cada selección nacional, con estadísticas acumuladas siguiendo el formato de **FBref**, además de un conjunto de métricas del rival (identificadas con el sufijo **`_against`**) para cada estadística.
* **`players`**: contiene un registro por cada jugador, con estadísticas individuales estándar. Los porteros incluyen un bloque específico de métricas identificado con el prefijo **`gk_*`**.

**Nota sobre la procedencia de los datos:** El esquema de la base de datos sigue las convenciones de **FBref** (Opta / Stats Perform). No incluye métricas avanzadas como **goles esperados (xG)** ni **pases progresivos**, ya que estas estadísticas dejaron de estar disponibles en FBref a partir de **enero de 2026**. Por ello, el conjunto de datos se limita a estadísticas básicas de conteo. Durante las primeras jornadas del torneo, cuando cada selección ha disputado solo uno o dos partidos, las tasas y métricas **por 90 minutos (per90)** deben interpretarse como indicios preliminares y no como tendencias definitivas.


## 1. ¿Por qué este conjunto de datos?

Una Copa Mundial concentra en apenas un mes toda la narrativa que normalmente se desarrolla a lo largo de una temporada: selecciones anfitrionas bajo presión, equipos debutantes que buscan sorprender y planteamientos tácticos enfrentándose a la realidad del campo. Gracias a la información detallada sobre **posesión del balón, remates, disciplina, jugadas a balón parado y rendimiento de los porteros** para cada selección, además de las estadísticas completas de cada jugador, este conjunto de datos permite responder preguntas concretas, tales como:

* ¿La posesión del balón se traduce en más goles?
* ¿Qué jugadores son más eficientes al momento de definir?
* ¿Qué porteros soportan una mayor carga de trabajo?
* ¿Qué tan profunda es la rotación de las plantillas?
* ¿Qué selecciones presentan estilos de juego similares?

Además, como el conjunto de datos **se actualiza diariamente**, estas mismas preguntas pueden volver a analizarse conforme aumenta la cantidad de información disponible, pasando de la primera jornada hasta la conclusión del torneo.

Durante las primeras fases del campeonato, cuando aún se han disputado pocos encuentros, resulta especialmente interesante observar la diferencia entre el **dominio mostrado en el juego** y los **resultados obtenidos**, ya que esa brecha suele ser más evidente con muestras de datos reducidas.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

sns.set_theme(style="whitegrid", context="notebook")
pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected"   # load plotly.js from a CDN so figures render in the published notebook
PLOTLY_KW = dict(template="plotly_white")

DATA_DIR = "dataset"
def _find(fname):
    bases = [DATA_DIR, "dataset", "."] + sorted(glob.glob("/kaggle/input/*"))
    for b in bases:
        hits = glob.glob(os.path.join(b, "**", fname), recursive=True)
        if hits:
            return sorted(hits, key=len)[0]
    raise FileNotFoundError(f"{fname} not found. Set DATA_DIR to the folder holding matches/players/teams.")
MATCHES_PATH, PLAYERS_PATH, TEAMS_PATH = _find("matches.csv"), _find("players.csv"), _find("teams.csv")

matches = pd.read_csv(MATCHES_PATH)
players = pd.read_csv(PLAYERS_PATH)
teams   = pd.read_csv(TEAMS_PATH)

# --- Numeric coercion. The feed is refreshed daily and adds upcoming fixtures
#     with blank stats, so never assume a column is fully populated. ---
def _num(s):
    """Coerce a possibly comma-grouped / blank string column to float; blanks -> NaN."""
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).str.replace("–", "-", regex=False),
        errors="coerce")

for c in ["home_score", "away_score"]:
    if c in matches:
        matches[c] = _num(matches[c])
matches["attendance_n"] = _num(matches["attendance"]) if "attendance" in matches else np.nan
# A fixture is "played" once both scores are present. Result-based charts use this subset.
matches["is_played"] = matches["home_score"].notna() & matches["away_score"].notna()
played_matches = matches[matches["is_played"]].copy()

# --- Derived player fields ---
def parse_age(a):
    """FBref stores age as 'YY-DDD' (years-days). Return float years."""
    if pd.isna(a): return np.nan
    s = str(a)
    if "-" in s:
        y, d = s.split("-"); return int(y) + int(d) / 365.0
    try: return float(s)
    except ValueError: return np.nan
players["age_years"] = players["age"].map(parse_age)
players["pos_main"]  = players["position"].str.split(",").str[0]            # primary position token
players["played"]    = players["games"].fillna(0) > 0                       # featured in any match so far
players["young_player_eligible"] = players["birth_year"] >= 2005            # FIFA Young Player Award: born >= 1 Jan 2005
played = players[players["played"]].copy()                                  # players who have appeared
gks    = players[players["gk_minutes"].notna()].copy()                      # goalkeepers with minutes

# Captains, read off the match sheets (blanks from unplayed fixtures dropped)
CAPTAINS = set(matches["home_captain"].dropna()) | set(matches["away_captain"].dropna())
players["is_captain"] = players["player"].isin(CAPTAINS)

# --- Stable team colours, derived from the teams file so new teams auto-colour ---
TEAM_ORDER = sorted(teams["team"].dropna().unique())
_pal = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24   # 48 distinct hues
TEAM_COLOR = {t: _pal[i % len(_pal)] for i, t in enumerate(TEAM_ORDER)}

# --- Group draw (reference map). Lookups are TOLERANT: a team missing from the
#     map resolves to "?" and drops out of group-only views, so the notebook
#     never crashes as the squad list grows toward all 48 teams. Extend GROUP
#     with the official draw when you have it. ---
GROUP = {
    "Mexico": "A", "South Africa": "A", "Korea Republic": "A", "Czechia": "A",
    "Canada": "B", "Switzerland": "B", "Qatar": "B", "Bosnia-Herzegovina": "B",
    "Brazil": "C", "Morocco": "C", "Haiti": "C", "Scotland": "C",
    "United States": "D", "Paraguay": "D", "Australia": "D", "Türkiye": "D",
}
def group_of(team):
    return GROUP.get(team, "?")
teams["group"]   = teams["team"].map(group_of)
players["group"] = players["team"].map(group_of)
GROUP_ORDER = sorted(g for g in teams["group"].unique() if g != "?")
_ungrouped = sorted(set(teams["team"]) - set(GROUP))
if _ungrouped:
    print(f"note: {len(_ungrouped)} team(s) not in the GROUP draw map (excluded from group views): "
          + ", ".join(_ungrouped))

# --- Host-city coordinates for the venue map. TOLERANT lookup: an unmapped city
#     yields (nan, nan) and is simply dropped from the map instead of raising a
#     KeyError. Covers all 16 FIFA WC 2026 host cities so the map stays complete
#     as the schedule moves between venues. ---
CITY_GEO = {
    # Mexico
    "Coyoacán": (19.350, -99.162), "Zapopan": (20.681, -103.462),
    "Guadalupe": (25.669, -100.244), "Monterrey": (25.686, -100.316),
    "Mexico City": (19.433, -99.133), "Guadalajara": (20.659, -103.349),
    # Canada
    "Toronto": (43.633, -79.418), "Vancouver": (49.277, -123.112),
    # United States
    "Inglewood": (33.953, -118.339), "Los Angeles": (34.052, -118.244),
    "Santa Clara": (37.403, -121.970), "East Rutherford": (40.814, -74.074),
    "Foxborough": (42.091, -71.264), "Atlanta": (33.755, -84.401),
    "Arlington": (32.747, -97.094), "Dallas": (32.777, -96.797),
    "Houston": (29.685, -95.411), "Kansas City": (39.049, -94.484),
    "Miami Gardens": (25.958, -80.239), "Miami": (25.762, -80.192),
    "Philadelphia": (39.901, -75.168), "Seattle": (47.595, -122.332),
}
def city_latlon(city):
    """Return (lat, lon) for a venue city, or (nan, nan) if unmapped. Never raises."""
    return CITY_GEO.get(city, (np.nan, np.nan))

# --- One row per team-per-match, built from the wide match table.
#     Only played fixtures contribute, and stats are read with .get so a missing
#     column degrades to NaN rather than a KeyError. ---
def matches_long(m):
    rows = []
    for _, r in m[m["is_played"]].iterrows():
        for side, opp in [("home", "away"), ("away", "home")]:
            d = {"team": r[f"{side}_team"], "opponent": r[f"{opp}_team"], "side": side,
                 "gf": int(r[f"{side}_score"]), "ga": int(r[f"{opp}_score"])}
            for st in ["possession", "sot", "total_shots", "saves", "cards_yellow",
                       "cards_red", "fouls", "corners", "crosses", "interceptions", "offsides"]:
                d[st] = r.get(f"{side}_{st}", np.nan)
            d["result"] = "W" if d["gf"] > d["ga"] else "L" if d["gf"] < d["ga"] else "D"
            d["points"] = 3 if d["result"] == "W" else 1 if d["result"] == "D" else 0
            d["attendance"] = r.get("attendance_n", np.nan)
            d["venue"] = r.get("venue"); d["referee"] = r.get("referee"); d["date"] = r.get("date")
            d["round"] = r.get("round"); d["gameweek"] = r.get("gameweek")
            rows.append(d)
    cols = ["team", "opponent", "side", "gf", "ga", "possession", "sot", "total_shots",
            "saves", "cards_yellow", "cards_red", "fouls", "corners", "crosses",
            "interceptions", "offsides", "result", "points", "attendance", "venue",
            "referee", "date", "round", "gameweek"]
    return pd.DataFrame(rows, columns=cols)   # typed-empty when no match is played yet
team_match = matches_long(matches)
team_match["group"] = team_match["team"].map(group_of)

# --- One row per team, aggregated across every match played so far. Correct
#     whether a team has 1 game or many, so downstream joins never assume a
#     single match per team. ---
def aggregate_team_match(tmf):
    cols = ["team", "P", "W", "D", "L", "points", "gf", "ga", "gd", "group"]
    if tmf.empty:
        return pd.DataFrame(columns=cols)
    g = (tmf.assign(W=lambda d: (d["result"] == "W").astype(int),
                    D=lambda d: (d["result"] == "D").astype(int),
                    L=lambda d: (d["result"] == "L").astype(int))
            .groupby("team")
            .agg(P=("result", "size"), W=("W", "sum"), D=("D", "sum"), L=("L", "sum"),
                 points=("points", "sum"), gf=("gf", "sum"), ga=("ga", "sum"))
            .reset_index())
    g["gd"] = g["gf"] - g["ga"]
    g["group"] = g["team"].map(group_of)
    return g[["team", "P", "W", "D", "L", "points", "gf", "ga", "gd", "group"]]
team_agg = aggregate_team_match(team_match)

# --- Dynamic scope counts. Titles below interpolate these so the notebook
#     re-labels itself automatically on every daily refresh. ---
N_MATCHES_TOTAL  = int(matches.shape[0])
N_MATCHES_PLAYED = int(played_matches.shape[0])
N_TEAMS   = int(teams.shape[0])
N_PLAYERS = int(players.shape[0])
N_PLAYED  = int(played.shape[0])
N_GK      = int(gks.shape[0])
N_TEAM_MATCH = int(team_match.shape[0])
ROUNDS = [r for r in played_matches["round"].dropna().unique()] if "round" in played_matches else []
MAX_GAMES = int(team_agg["P"].max()) if not team_agg.empty else 0
SCOPE = f"{N_MATCHES_PLAYED} matches played"   # short slice label reused in chart titles

print(f"matches {matches.shape} ({N_MATCHES_PLAYED} played, "
      f"{N_MATCHES_TOTAL - N_MATCHES_PLAYED} scheduled) | players {players.shape} | teams {teams.shape}")
print(f"{N_PLAYED} players featured, {N_GK} keepers, {N_TEAM_MATCH} team-match rows, "
      f"up to {MAX_GAMES} game(s) per team")
print("rounds so far:", ", ".join(ROUNDS) if ROUNDS else "none")
